# Step 14 — Transformer comparison: TF-IDF vs BERT vs RoBERTa

**Question.** Does a context-aware encoder (BERT, RoBERTa) beat the TF-IDF + logistic-regression detector of Step 13 on the same PESO silver labels?

**Design (identical to Step 13 unless stated).** Same 7,280 in-scope units, same `GroupKFold(5)` by employer, same employer-bootstrap weights (seed 42, 1,000 resamples).
Transformers are fine-tuned for 3-class (none / implicit / explicit); the biased-vs-not result is derived as class > 0.

**Transformer protocol (fixed in advance, not tuned on test folds).**
- Models: `bert-base-uncased`, `roberta-base` (revisions recorded in `step14_environment.json`).
- Training texts are de-duplicated within each training fold; all implicit and explicit texts are kept, and `none` is randomly under-sampled to 1,500 distinct texts (CPU budget).
- Loss: cross-entropy with balanced class weights computed on that training subset. 3 epochs, batch 16, AdamW lr 3e-5, 10% warm-up, linear decay, max length 64, seed 42.
- Prediction is made on every test unit (de-duplicated for speed, then mapped back).
- Variants: full text, and keywords removed (the Step 13 stripping function) for every model.

**Not covered:** span localisation (only the TF-IDF token tagger exists); no hyper-parameter search; labels remain rule-derived silver labels, so scores measure reproduction of the rules, not validity.

In [1]:
import json, re, platform
from pathlib import Path
import numpy as np, pandas as pd, sklearn, joblib
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

SEED, N_SPLITS, N_BOOT = 42, 5, 1000
OUT = Path("outputs/equal_step14_transformer_comparison"); OUT.mkdir(parents=True, exist_ok=True)
json.dump(dict(python=platform.python_version(), platform=platform.platform(), numpy=np.__version__, pandas=pd.__version__,
               scikit_learn=sklearn.__version__, seed=SEED, cv="GroupKFold(5) by employer_id", bootstrap=f"{N_BOOT} resamples of employers",
               text_model="TfidfVectorizer(1-2 grams, min_df=2, sublinear_tf) + LogisticRegression(class_weight=balanced, max_iter=2000)",
               tagger="DictVectorizer on word/prefix/suffix/shape/neighbour features + LogisticRegression(class_weight=balanced), threshold 0.5"),
          open(OUT / "step14_tfidf_environment.json", "w"), indent=2)

In [2]:
MASC = ("active adventurous aggress ambitio analy assert athlet autonom boast challeng compet confident "
        "courag decisive decision determin dominan driven fearless force headstrong hierarch impulsive "
        "independen individual intellect lead logic objective opinion outspoken persist self-relian "
        "self-sufficien stubborn superior unreasonab").split()
FEM = ("agree affectionate child cheer collab commit communal compassion connect considerate cooperat "
       "depend emotiona empath feel flatterable gentle honest interpersonal interdependen kind kinship "
       "loyal modest nurtur pleasant polite quiet respons sensitiv submissive support sympath tender "
       "together trust understand warm enthusias inclusive yield share").split()
FP = ("confidential", "committee", "shareholder", "kinds")
DUTY = ("analy", "lead", "respons", "support", "decision", "challeng", "collab", "cooperat")

def coded_hits(text, stems):
    out = []
    for m in re.finditer(r"[a-z][a-z-]*", text.lower()):
        w = m.group()
        if w.startswith(FP): continue
        ps = [w] + (w.split("-") if "-" in w else [])
        if any(p.startswith(s) for p in ps for s in stems): out.append((m.start(), m.end(), w))
    return out

MALE = {"std": r"male|males|men|gentlemen", "lem": r"male|males|men|man|gentlemen|gentleman"}
FEMALE = {"std": r"female|females|women|ladies", "lem": r"female|females|women|woman|ladies|lady"}
INCL_RE = re.compile(r"\b(?:either\s+gender|any\s+gender|regardless\s+of\s+(?:sex|gender)|men\s+and\s+women|"
                     r"gender\s*[:\-]?\s*(?:any|open|not\s+required))\b", re.I)
TITLE_RE = re.compile(r"\b(?:salesman|salesmen|saleslady|salesgirl|waiter|waitress|chairman|foreman|watchman|cameraman|busboy|houseboy|stewardess)\b", re.I)
PRON_BIN_RE = re.compile(r"\b(?:he\s*/\s*she|she\s*/\s*he|he\s+or\s+she|his\s*/\s*her|s/he)\b", re.I)
PRON_RE = re.compile(r"\b(?:he|she|his|her|him)\b", re.I)
ONLY_RE = re.compile(r"\b(?:only|exclusively|must\s+be|should\s+be|required)\b", re.I)
PREF_RE = re.compile(r"\b(?:prefer|preferably|preferred|preference)\b", re.I)

def layerA(text, lemma=False):
    k = "lem" if lemma else "std"
    m_re, f_re = re.compile(r"\b(?:%s)\b" % MALE[k], re.I), re.compile(r"\b(?:%s)\b" % FEMALE[k], re.I)
    m, f = m_re.search(text), f_re.search(text)
    n_words = len(re.findall(r"\w+", text))
    out = dict(gender_reference="no", gender_expression_type="none", gender_direction="na", gender_coding="na", span=None, context_dependent="no")
    inc, pb = INCL_RE.search(text), PRON_BIN_RE.search(text)
    if inc:
        out.update(gender_reference="yes", gender_expression_type="inclusive", gender_direction="multiple", span=inc.span())
    elif m and f:
        s = min(m.start(), f.start()); e = max(m.end(), f.end())
        out.update(gender_reference="yes", gender_expression_type="binary_listing", gender_direction="multiple", span=(s, e))
    elif m or f:
        h = m or f; d = "male" if m else "female"
        t = "restriction" if (n_words <= 4 or ONLY_RE.search(text)) and not PREF_RE.search(text) else ("preference" if PREF_RE.search(text) else "unclear")
        out.update(gender_reference="yes", gender_expression_type=t, gender_direction=d, span=h.span())
    elif pb:
        out.update(gender_reference="yes", gender_expression_type="binary_listing", gender_direction="multiple", span=pb.span())
    elif TITLE_RE.search(text) or PRON_RE.search(text):
        h = TITLE_RE.search(text) or PRON_RE.search(text)
        w = h.group().lower(); d = "male" if w in ("salesman", "salesmen", "waiter", "chairman", "foreman", "watchman", "cameraman", "busboy", "houseboy", "he", "his", "him") else "female"
        out.update(gender_reference="yes", gender_expression_type="gendered_title_pronoun", gender_direction=d, span=h.span())
    else:
        mh, fh = coded_hits(text, MASC), coded_hits(text, FEM)
        if mh or fh:
            allh = sorted(mh + fh)
            out.update(gender_expression_type="gender_coded", gender_coding="agentic" if mh and not fh else "communal" if fh and not mh else "mixed",
                       span=(allh[0][0], allh[0][1]), context_dependent="yes" if all(w.startswith(DUTY) for *_, w in allh) else "no")
    return out

BMAP = {  # Layer A type -> (bias_type, bias_subtype, rewrite_action)
    "restriction": ("explicit", "gender_exclusion", "required"), "preference": ("explicit", "gender_preference", "required"),
    "binary_listing": ("unclear", "unclear", "recommended"), "gendered_title_pronoun": ("implicit", "gendered_role_language", "recommended"),
    "gender_coded": ("unclear", "unclear", "manual_review"), "inclusive": ("none", "none", "not_needed"), "none": ("none", "none", "not_needed"),
    "unclear": ("unclear", "unclear", "manual_review")}
PRESENCE = {"explicit": "biased", "implicit": "biased", "none": "not_biased", "unclear": "unclear"}

def preannotate(text, lemma=False):
    a = layerA(text, lemma)
    bt, st, ra = BMAP[a["gender_expression_type"]]
    rewrite = "Open to all qualified candidates" if (bt == "explicit" or a["gender_expression_type"] == "binary_listing") and len(re.findall(r"\w+", text)) <= 4 else ""
    return dict(**a, bias_type=bt, bias_presence=PRESENCE[bt], bias_subtype=st, rewrite_action=ra, suggested_rewrite=rewrite)


In [3]:
units = pd.read_csv("outputs/equal_step4_sentence_embeddings/step4_sentence_units.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
rows = []
for s in units.sentence:
    r = preannotate(s); rows.append(dict(bias_type=r["bias_type"], layerA=r["gender_expression_type"], span=r["span"]))
d = pd.concat([units[["sentence_id", "record_id", "employer_id", "sentence"]], pd.DataFrame(rows)], axis=1)
print(d.bias_type.value_counts().to_string())
ds = d[d.bias_type != "unclear"].reset_index(drop=True)
CL = ["none", "implicit", "explicit"]
ds["y3"] = ds.bias_type.map({c: i for i, c in enumerate(CL)}); ds["yb"] = (ds.y3 > 0).astype(int)
print("\nin scope (unclear excluded):", len(ds), "| employers:", ds.employer_id.nunique())
summ = ds.groupby("bias_type").agg(units=("sentence", "size"), distinct_texts=("sentence", "nunique"), employers=("employer_id", "nunique")).loc[CL]
display(summ); summ.to_csv(OUT / "step14_tfidf_label_counts.csv")
print("Layer A types behind implicit:", ds[ds.bias_type == "implicit"].layerA.value_counts().to_dict())
print("implicit distinct texts:", sorted(ds[ds.bias_type == "implicit"].sentence.str.lower().unique())[:15])

bias_type
none        6929
unclear      691
explicit     313
implicit      38

in scope (unclear excluded): 7280 | employers: 320


,units,distinct_texts,employers
bias_type,,,
none,6929,4252,320
implicit,38,24,15
explicit,313,12,124


Layer A types behind implicit: {'gendered_title_pronoun': 38}
implicit distinct texts: ['booking salesman reliever', 'construction foreman', 'foreman', 'foreman (earthworks and civil)', 'foreman (earthworks/civil)', 'foreman civil', 'foreman electrical', 'foreman mechanical', 'foreman plumbing', 'general foreman', 'jr relief salesman', 'junior salesman', 'micro d salesman', 'saleslady', 'salesman']


In [4]:
groups = ds.employer_id.values; ue, inv = np.unique(groups, return_inverse=True); G = len(ue)
folds = list(GroupKFold(n_splits=N_SPLITS).split(ds, ds.y3, groups))
rng = np.random.default_rng(SEED); W = rng.multinomial(G, np.ones(G) / G, size=N_BOOT)

def make(): return LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
def vec(): return TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True)

def oof(texts, y, frame, folds):
    pred = np.zeros(len(frame), int)
    for tr, te in folds:
        v = vec(); m = make().fit(v.fit_transform(texts[tr]), y[tr]); pred[te] = m.predict(v.transform(texts[te]))
    return pred

def cm_groups(y, p, K, inv, G):
    out = np.zeros((G, K, K)); np.add.at(out, (inv, y, p), 1); return out
def prf(cm):
    tp = np.diag(cm); p = np.divide(tp, cm.sum(0), out=np.zeros(len(cm)), where=cm.sum(0) > 0)
    r = np.divide(tp, cm.sum(1), out=np.zeros(len(cm)), where=cm.sum(1) > 0)
    f = np.divide(2 * p * r, p + r, out=np.zeros(len(cm)), where=(p + r) > 0); return p, r, f
def report(y, p, K, names, inv_, G_, W_):
    cmg = cm_groups(y, p, K, inv_, G_); cm = cmg.sum(0); P, R, F = prf(cm)
    bs = np.array([[*prf(np.tensordot(w, cmg, axes=(0, 0)))[2], prf(np.tensordot(w, cmg, axes=(0, 0)))[2].mean()] for w in W_])
    ci = lambda k: f"{np.percentile(bs[:, k], 2.5):.3f}-{np.percentile(bs[:, k], 97.5):.3f}"
    t = pd.DataFrame({"precision": P.round(3), "recall": R.round(3), "F1": F.round(3), "F1_95CI": [ci(k) for k in range(K)], "support": cm.sum(1).astype(int)}, index=names)
    t.loc["macro"] = [P.mean().round(3), R.mean().round(3), F.mean().round(3), ci(K), int(cm.sum())]
    return t, pd.DataFrame(cm.astype(int), index=[f"true_{n}" for n in names], columns=[f"pred_{n}" for n in names])

texts = ds.sentence.values
res = {}
p3 = oof(texts, ds.y3.values, ds, folds); pb = oof(texts, ds.yb.values, ds, folds)
res["3class"] = report(ds.y3.values, p3, 3, CL, inv, G, W); res["binary"] = report(ds.yb.values, pb, 2, ["not_biased", "biased"], inv, G, W)
print("=== Gender-bias detection (binary model) ==="); display(res["binary"][0]); display(res["binary"][1])
print("=== Explicit / implicit classification (3-class model) ==="); display(res["3class"][0]); display(res["3class"][1])
for k, (t, c) in res.items(): t.to_csv(OUT / f"step14_tfidf_{k}_metrics.csv"); c.to_csv(OUT / f"step14_tfidf_{k}_confusion.csv")
print("chance reference, always 'none': macro-F1 =", round(prf(np.array([[len(ds) - (ds.y3 > 0).sum(), 0, 0], [(ds.y3 == 1).sum(), 0, 0], [(ds.y3 == 2).sum(), 0, 0]]))[2].mean(), 3))
ds["pred3"] = [CL[i] for i in p3]; ds["predb"] = pb
ds[ds.pred3 != ds.bias_type][["sentence_id", "employer_id", "sentence", "bias_type", "pred3"]].to_csv(OUT / "step14_tfidf_oof_errors.csv", index=False)
print("out-of-fold errors saved:", int((ds.pred3 != ds.bias_type).sum()))

=== Gender-bias detection (binary model) ===


,precision,recall,F1,F1_95CI,support
not_biased,0.998,1.000,0.999,0.998-1.000,6929
biased,1.000,0.957,0.978,0.954-0.995,351
macro,0.999,0.979,0.989,0.976-0.998,7280


,pred_not_biased,pred_biased
true_not_biased,6929,0
true_biased,15,336


=== Explicit / implicit classification (3-class model) ===


,precision,recall,F1,F1_95CI,support
none,0.999,1.000,0.999,0.999-1.000,6929
implicit,0.921,0.921,0.921,0.667-0.982,38
explicit,1.000,0.990,0.995,0.987-1.000,313
macro,0.973,0.970,0.972,0.889-0.992,7280


,pred_none,pred_implicit,pred_explicit
true_none,6926,3,0
true_implicit,3,35,0
true_explicit,3,0,310


chance reference, always 'none': macro-F1 = 0.325
out-of-fold errors saved: 9


In [5]:
import time, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from huggingface_hub import model_info
torch.set_num_threads(8)
MODELS = ["bert-base-uncased", "roberta-base"]
CACHE = OUT / "cache"; CACHE.mkdir(exist_ok=True)
EPOCHS, BATCH, LR, MAXLEN, N_NONE = 3, 16, 3e-5, 64, 1500
env = json.load(open(OUT / "step14_tfidf_environment.json")) if (OUT / "step14_tfidf_environment.json").exists() else {}
env.update(torch=torch.__version__, transformers=__import__("transformers").__version__,
           revisions={m: model_info(m).sha for m in MODELS}, epochs=EPOCHS, batch=BATCH, lr=LR, max_len=MAXLEN, none_cap=N_NONE)
json.dump(env, open(OUT / "step14_environment.json", "w"), indent=2); print(env["revisions"])

def strip(s):
    for rx in (INCL_RE, TITLE_RE, PRON_BIN_RE, PRON_RE, re.compile(r"\b(?:%s)\b" % MALE["std"], re.I), re.compile(r"\b(?:%s)\b" % FEMALE["std"], re.I)):
        s = rx.sub(" ", s)
    return s
VARIANTS = {"full": texts, "keywords_removed": np.array([strip(s) for s in texts])}
y3 = ds.y3.values

def fit_predict(model_name, tx, tr, te, k):
    torch.manual_seed(SEED); rs = np.random.default_rng(SEED + k)
    d = pd.DataFrame({"t": tx[tr], "y": y3[tr]}).drop_duplicates("t")
    none = d[d.y == 0]; keep = pd.concat([d[d.y > 0], none.iloc[rs.permutation(len(none))[:N_NONE]]]).sample(frac=1, random_state=SEED + k)
    cnt = np.bincount(keep.y, minlength=3); cw = torch.tensor(len(keep) / (3 * np.maximum(cnt, 1)), dtype=torch.float)
    tok = AutoTokenizer.from_pretrained(model_name); m = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * int(np.ceil(len(keep) / BATCH)); sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    lossf = torch.nn.CrossEntropyLoss(weight=cw); T, Y = keep.t.tolist(), keep.y.values
    m.train()
    for ep in range(EPOCHS):
        order = np.random.default_rng(SEED + 100 * k + ep).permutation(len(T))
        for b in range(0, len(T), BATCH):
            ix = order[b:b + BATCH]; x = tok([T[i] for i in ix], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN)
            loss = lossf(m(**x).logits, torch.tensor(Y[ix])); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sch.step(); opt.zero_grad()
    m.eval(); ut = list(dict.fromkeys(tx[te])); out = {}
    with torch.no_grad():
        for b in range(0, len(ut), 128):
            x = tok(ut[b:b + 128], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN)
            for s, p in zip(ut[b:b + 128], m(**x).logits.argmax(-1).tolist()): out[s] = p
    return np.array([out[s] for s in tx[te]]), len(keep)

log = []
for var in VARIANTS:
    for mn in MODELS:
        for k, (tr, te) in enumerate(folds):
            f = CACHE / f"{mn.replace('/', '_')}_{var}_fold{k}.npy"
            if f.exists(): continue
            t0 = time.time(); pr, ntr = fit_predict(mn, VARIANTS[var], tr, te, k); np.save(f, pr)
            print(f"{mn} | {var} | fold {k} | train texts {ntr} | {time.time() - t0:.0f}s | fold macro-acc {(pr == y3[te]).mean():.4f}", flush=True)

D:\Dev\project-equal\equal-v2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'bert-base-uncased': '86b5e0934494bd15c9632b12f734a8a67f723594', 'roberta-base': 'e2da8e2f811d1448a5b465c236feacd80ffbac7b'}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  30%|██▉       | 59/199 [00:00<00:00, 537.04it/s]

Loading weights:  57%|█████▋    | 113/199 [00:00<00:00, 424.93it/s]

...

Loading weights:  12%|█▏        | 24/197 [00:00<00:00, 228.74it/s]

Loading weights:  24%|██▍       | 47/197 [00:00<00:00, 209.39it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 693.46it/s]


[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


roberta-base | keywords_removed | fold 4 | train texts 1528 | 665s | fold macro-acc 0.9560


Collect cached out-of-fold predictions and compare against TF-IDF. Differences are computed on the same employer-bootstrap resamples (paired), so the interval on the difference accounts for employer clustering.

In [6]:
def load(mn, var):
    p = np.zeros(len(ds), int)
    for k, (tr, te) in enumerate(folds): p[te] = np.load(CACHE / f"{mn}_{var}_fold{k}.npy")
    return p
P = {("tfidf", "full"): p3, ("tfidf", "keywords_removed"): oof(VARIANTS["keywords_removed"], y3, ds, folds)}
for mn in MODELS:
    for var in VARIANTS: P[(mn, var)] = load(mn, var)

def stats(p):
    c3 = cm_groups(y3, p, 3, inv, G); cb = cm_groups((y3 > 0).astype(int), (p > 0).astype(int), 2, inv, G)
    return c3, cb
def scalars(c3, cb):
    f3 = prf(c3)[2]; pb_, rb_, fb_ = prf(cb); return np.array([pb_[1], rb_[1], fb_[1], f3[1], f3[2], f3.mean()])
NAMES = ["biased_P", "biased_R", "biased_F1", "F1_implicit", "F1_explicit", "macro_F1"]
S = {key: stats(p) for key, p in P.items()}
def boot(key): c3, cb = S[key]; return np.array([scalars(np.tensordot(w, c3, (0, 0)), np.tensordot(w, cb, (0, 0))) for w in W])
B = {key: boot(key) for key in S}
rows = []
for (mn, var), (c3, cb) in S.items():
    v = scalars(c3.sum(0), cb.sum(0)); r = dict(model=mn, variant=var, **{n: round(x, 3) for n, x in zip(NAMES, v)})
    r["macro_F1_CI"] = f"{np.percentile(B[(mn, var)][:, 5], 2.5):.3f}-{np.percentile(B[(mn, var)][:, 5], 97.5):.3f}"
    if mn != "tfidf":
        d = B[(mn, var)][:, 5] - B[("tfidf", var)][:, 5]
        r["macro_F1_diff_vs_tfidf"] = round(v[5] - scalars(*(x.sum(0) for x in S[("tfidf", var)]))[5], 3)
        r["diff_95CI"] = f"{np.percentile(d, 2.5):.3f} to {np.percentile(d, 97.5):.3f}"
    rows.append(r)
cmp_ = pd.DataFrame(rows); display(cmp_); cmp_.to_csv(OUT / "step14_comparison.csv", index=False)
for (mn, var), p in P.items():
    pd.DataFrame(S[(mn, var)][0].sum(0).astype(int), index=[f"true_{c}" for c in CL], columns=[f"pred_{c}" for c in CL]).to_csv(OUT / f"step14_confusion_{mn}_{var}.csv")
    print(f"--- {mn} | {var} ---"); display(pd.DataFrame(S[(mn, var)][0].sum(0).astype(int), index=[f"true_{c}" for c in CL], columns=[f"pred_{c}" for c in CL]))

,model,variant,biased_P,biased_R,biased_F1,F1_implicit,F1_explicit,macro_F1,macro_F1_CI,macro_F1_diff_vs_tfidf,diff_95CI
0,tfidf,full,0.991,0.983,0.987,0.921,0.995,0.972,0.889-0.992,NaN,NaN
1,tfidf,keywords_removed,0.402,0.954,0.565,0.107,0.557,0.542,0.502-0.574,NaN,NaN
2,bert-base-uncased,full,0.991,0.980,0.986,0.873,0.998,0.957,0.898-0.988,-0.015,-0.059 to 0.031
3,bert-base-uncased,keywords_removed,0.943,0.761,0.842,0.069,0.000,0.354,0.340-0.369,-0.188,-0.226 to -0.143
4,roberta-base,full,0.934,0.803,0.864,0.404,0.787,0.728,0.601-0.831,-0.244,-0.369 to -0.140
5,roberta-base,keywords_removed,1.000,0.994,0.997,0.196,0.000,0.399,0.354-0.445,-0.143,-0.196 to -0.085


--- tfidf | full ---


,pred_none,pred_implicit,pred_explicit
true_none,6926,3,0
true_implicit,3,35,0
true_explicit,3,0,310


--- tfidf | keywords_removed ---


,pred_none,pred_implicit,pred_explicit
true_none,6430,33,466
true_implicit,12,4,22
true_explicit,4,0,309


...

,pred_none,pred_implicit,pred_explicit
true_none,6913,16,0
true_implicit,27,11,0
true_explicit,57,256,0


--- roberta-base | full ---


,pred_none,pred_implicit,pred_explicit
true_none,6909,2,18
true_implicit,11,19,8
true_explicit,58,35,220


--- roberta-base | keywords_removed ---


,pred_none,pred_implicit,pred_explicit
true_none,6929,0,0
true_implicit,0,38,0
true_explicit,2,311,0


Where the models disagree with the silver labels (full text), and the Step 13 demo sentences.

In [7]:
err = ds[["sentence_id", "employer_id", "sentence", "bias_type"]].copy()
for mn in ["tfidf"] + MODELS: err[mn] = [CL[i] for i in P[(mn, "full")]]
bad = err[(err[["tfidf"] + MODELS].ne(err.bias_type, axis=0)).any(axis=1)]
bad.to_csv(OUT / "step14_errors_full.csv", index=False)
print("units with at least one model error (full text):", len(bad), "| per model:", {m: int((err[m] != err.bias_type).sum()) for m in ["tfidf"] + MODELS})
display(bad.drop_duplicates("sentence").head(40))

# demo with a final fit on all data is skipped for the transformers (CPU cost); cached fold models are not stored.
print("Distinct-text agreement none/implicit/explicit counts of silver labels:", ds.drop_duplicates("sentence").bias_type.value_counts().to_dict())

units with at least one model error (full text): 137 | per model: {'tfidf': 9, 'bert-base-uncased': 10, 'roberta-base': 132}


,sentence_id,employer_id,sentence,bias_type,tfidf,bert-base-uncased,roberta-base
119,row-0026-s002,EMP-8286a68d,Male,explicit,explicit,explicit,none
405,row-0083-s001,EMP-7f9fab77,Watchman,implicit,none,none,implicit
634,row-0131-s001,EMP-bd1ecdbc,Foreman,implicit,implicit,implicit,none
803,row-0176-s001,EMP-04c3c422,Saleslady,implicit,none,none,none
837,row-0201-s001,EMP-b7808be6,Foreman (Earthworks and Civil),implicit,implicit,implicit,none
868,row-0220-s004,EMP-2ad25376,Single,none,none,none,explicit
878,row-0221-s006,EMP-2ad25376,With or without experience as Stockman,none,none,none,implicit
897,row-0236-s003,EMP-74cf88c1,Single or Married,none,none,none,explicit
934,row-0264-s001,EMP-bef15e94,Warehouseman,none,none,none,explicit
935,row-0265-s001,EMP-bef15e94,Foreman (Earthworks/Civil),implicit,implicit,implicit,none


Distinct-text agreement none/implicit/explicit counts of silver labels: {'none': 4252, 'implicit': 24, 'explicit': 12}
